In [11]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import time
import numpy as np
import os

In [12]:
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
 
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
encoder = SentenceTransformer("all-MiniLM-L6-v2")  



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
# Test configurations
configs = [
    {"name": "fast_initial_upload", "m": 0, "ef_construct": 100},  # m=0 = ingest-only
    {"name": "memory_optimized", "m": 8, "ef_construct": 100},  # m=8 = lower RAM
    {"name": "balanced", "m": 16, "ef_construct": 200},  # m=16 = balanced
    {"name": "high_quality", "m": 32, "ef_construct": 400},  # m=32 = higher recall, slower build
]

for config in configs:
    collection_name = f"my_domain_{config['name']}"
    if client.collection_exists(collection_name=collection_name):
        client.delete_collection(collection_name=collection_name)

    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
        hnsw_config=models.HnswConfigDiff(
            m=config["m"],
            ef_construct=config["ef_construct"],
            full_scan_threshold=10,  # force HNSW instead of full scan
        ),
        optimizers_config=models.OptimizersConfigDiff(
            indexing_threshold=10
        ),  # Force indexing even on small sets for demo
    )
    print(f"Created collection: {collection_name}")

Created collection: my_domain_fast_initial_upload
Created collection: my_domain_memory_optimized
Created collection: my_domain_balanced
Created collection: my_domain_high_quality


In [16]:
def upload_with_timing(collection_name, data, config_name):
    embeddings = encoder.encode(
        [d["description"] for d in data],
        show_progress_bar=True,
    ).tolist()

    points = []
    for i, item in enumerate(data):
        description = item["description"]

        points.append(
            models.PointStruct(
                id=i,
                vector=embeddings[i],
                payload={
                    **item,
                    "length": len(description),
                    "word_count": len(description.split()),
                    "has_keywords": any(
                        keyword in description.lower()
                        for keyword in ["important", "key", "main"]
                    ),
                },
            )
        )

    start_time = time.time()
    client.upload_points(collection_name=collection_name, points=points)
    upload_time = time.time() - start_time

    # Warmup after upload
    client.query_points(
        collection_name=collection_name,
        query=points[0].vector,
        limit=1,
    )

    print(f"{config_name}: Uploaded {len(points)} points in {upload_time:.2f}s")
    return upload_time

# Load your dataset here. The larger the dataset, the more accurate the benchmark will be.
# your_dataset = [{"description": "This is a description of a product"}, ...]
your_dataset = [
    {
        "description": f"This is product {i}. It has important key features for vector search benchmarking."
    }
    for i in range(1000)
]
# Upload to each collection
upload_times = {}
for config in configs:
    collection_name = f"my_domain_{config['name']}"
    upload_times[config["name"]] = upload_with_timing(collection_name, your_dataset, config["name"])


def wait_for_indexing(collection_name, timeout=60, poll_interval=1):
    print(f"Waiting for collection '{collection_name}' to be indexed...")
    start_time = time.time()

    while time.time() - start_time < timeout:
        info = client.get_collection(collection_name=collection_name)

        if info.indexed_vectors_count > 0 and info.status == models.CollectionStatus.GREEN:
            print(f"Success! Collection '{collection_name}' is indexed and ready.")
            print(f" - Status: {info.status.value}")
            print(f" - Indexed vectors: {info.indexed_vectors_count}")
            return

        print(f" - Status: {info.status.value}, Indexed vectors: {info.indexed_vectors_count}. Waiting...")
        time.sleep(poll_interval)

    info = client.get_collection(collection_name=collection_name)
    raise Exception(
        f"Timeout reached after {timeout} seconds. Collection '{collection_name}' is not ready. "
        f"Final status: {info.status.value}, Indexed vectors: {info.indexed_vectors_count}"
    )


for config in configs:
    if config["m"] > 0:  # m=0 has no HNSW to wait for
        collection_name = f"my_domain_{config['name']}"
        wait_for_indexing(collection_name)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

fast_initial_upload: Uploaded 1000 points in 1.32s


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

memory_optimized: Uploaded 1000 points in 1.01s


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

balanced: Uploaded 1000 points in 1.14s


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

high_quality: Uploaded 1000 points in 1.10s
Waiting for collection 'my_domain_memory_optimized' to be indexed...
Success! Collection 'my_domain_memory_optimized' is indexed and ready.
 - Status: green
 - Indexed vectors: 1000
Waiting for collection 'my_domain_balanced' to be indexed...
Success! Collection 'my_domain_balanced' is indexed and ready.
 - Status: green
 - Indexed vectors: 1000
Waiting for collection 'my_domain_high_quality' to be indexed...
 - Status: yellow, Indexed vectors: 768. Waiting...
Success! Collection 'my_domain_high_quality' is indexed and ready.
 - Status: green
 - Indexed vectors: 1000


In [18]:
def benchmark_search(collection_name, query_embeddings, ef_values=[64, 128, 256], runs_per_query=5):
    results = {}

    # Warmup
    client.query_points(
        collection_name=collection_name,
        query=query_embeddings[0],
        limit=1,
        with_payload=False,
    )

    for hnsw_ef in ef_values:
        times = []

        for query_embedding in query_embeddings:
            for _ in range(runs_per_query):
                start_time = time.perf_counter()

                client.query_points(
                    collection_name=collection_name,
                    query=query_embedding,
                    limit=10,
                    search_params=models.SearchParams(hnsw_ef=hnsw_ef),
                    with_payload=False,
                )

                times.append((time.perf_counter() - start_time) * 1000)

        results[hnsw_ef] = {
            "avg_time": np.mean(times),
            "p95_time": np.percentile(times, 95),
            "min_time": np.min(times),
            "max_time": np.max(times),
        }

    return results


test_queries = [
    "important product features",
    "main benefit of fast search",
    "key vector database benchmark",
    "semantic search performance",
    "optimized indexing speed",
]

query_embeddings = encoder.encode(test_queries).tolist()

performance_results = {}

for config in configs:
    if config["m"] > 0:
        collection_name = f"my_domain_{config['name']}"
        performance_results[config["name"]] = benchmark_search(
            collection_name,
            query_embeddings,
        )

In [ ]:
def test_filtering_performance(collection_name):
    query_embedding = encoder.encode("your filter test query").tolist()

    # Test filter without index
    filter_condition = models.Filter(
        must=[models.FieldCondition(key="length", range=models.Range(gte=10, lte=200))]
    )

    # Demo only: unindexed_filtering_retrieve=True forces a scan; turn it off right after measuring.
    client.update_collection(
        collection_name=collection_name,
        strict_mode_config=models.StrictModeConfig(unindexed_filtering_retrieve=True),
    )

    # Warmup
    client.query_points(collection_name=collection_name, query=query_embedding, limit=1)

    # Timing without payload index
    times = []
    for _ in range(25):
        start_time = time.time()
        _ = client.query_points(
            collection_name=collection_name,
            query=query_embedding,
            query_filter=filter_condition,
            limit=10,
            with_payload=False,
        )
        times.append((time.time() - start_time) * 1000)
    time_without_index = np.mean(times)

    # Create payload index
    client.create_payload_index(
        collection_name=collection_name,
        field_name="length",
        field_schema=models.PayloadSchemaType.INTEGER,
        wait=True,
    )

    # HNSW was already built; adding the payload index doesn’t rebuild it.
    # Bump ef_construct (+1) once to trigger a safe rebuild.
    base_ef = client.get_collection(
        collection_name=collection_name
    ).config.hnsw_config.ef_construct
    new_ef_construct = base_ef + 1

    client.update_collection(
        collection_name=collection_name,
        hnsw_config=models.HnswConfigDiff(ef_construct=new_ef_construct),
        strict_mode_config=models.StrictModeConfig(
            unindexed_filtering_retrieve=False
        ),  # Turn off scanning and use payload index instead.
        
    )

    wait_for_indexing(collection_name)

    # Warmup
    client.query_points(collection_name=collection_name, query=query_embedding, limit=1)

    # Timing with index
    times = []
    for _ in range(25):
        start_time = time.time()
        _ = client.query_points(
            collection_name=collection_name,
            query=query_embedding,
            query_filter=filter_condition,
            limit=10,
            with_payload=False,
        )
        times.append((time.time() - start_time) * 1000)
    time_with_index = np.mean(times)

    return {
        "without_index": time_without_index,
        "with_index": time_with_index,
        "speedup": time_without_index / time_with_index,
    }


# Test on your best performing collection
best_collection = "my_domain_balanced"  # Choose based on your results
filtering_results = test_filtering_performance(best_collection)

Waiting for collection 'my_domain_balanced' to be indexed...
 - Status: yellow, Indexed vectors: 1000. Waiting...
Success! Collection 'my_domain_balanced' is indexed and ready.
 - Status: green
 - Indexed vectors: 1000


In [20]:
print("=" * 60)
print("PERFORMANCE OPTIMIZATION RESULTS")
print("=" * 60)

print("\n1) Upload Performance:")
for config_name, time_taken in upload_times.items():
    print(f"   {config_name}: {time_taken:.2f}s")

print("\n2) Search Performance (hnsw_ef=128):")
for config_name, results in performance_results.items():
    if 128 in results:
        print(f"   {config_name}: {results[128]['avg_time']:.2f}ms")

print("\n3) Filtering Impact:")
print(f"   Without index: {filtering_results['without_index']:.2f}ms")
print(f"   With index: {filtering_results['with_index']:.2f}ms")
print(f"   Speedup: {filtering_results['speedup']:.1f}x")

PERFORMANCE OPTIMIZATION RESULTS

1) Upload Performance:
   fast_initial_upload: 1.32s
   memory_optimized: 1.01s
   balanced: 1.14s
   high_quality: 1.10s

2) Search Performance (hnsw_ef=128):
   memory_optimized: 26.48ms
   balanced: 26.36ms
   high_quality: 26.74ms

3) Filtering Impact:
   Without index: 28.74ms
   With index: 27.93ms
   Speedup: 1.0x
